# Week 3 Crew Exercise: Build a CrewAI Agent System

This notebook is fully runnable cell-by-cell and stays in Week 3 scope (`3_crew`):
- `Agent`, `Task`, `Crew`, `Process`
- multi-agent coordination
- kickoff with runtime inputs


## 1. Install Dependencies (if needed)

If `crewai` is already installed, you can skip this cell.


In [ ]:
# %pip install -q crewai


## 2. Imports and Environment


In [ ]:
import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process

load_dotenv(override=True)

if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('OPENAI_API_KEY is missing in .env')

print('Environment loaded successfully.')


## 3. Define the Use Case

We will build a **Career Pivot Crew** that helps transition into a target role.


In [ ]:
inputs = {
    'target_role': 'AI Engineer',
    'region': 'East Africa',
    'background': 'Software engineer with Python and backend experience, limited ML deployment exposure.'
}

inputs


## 4. Define Agents


In [ ]:
market_researcher = Agent(
    role='Career Market Researcher',
    goal='Analyze market demand, core requirements, and hiring signals for {target_role} in {region}.',
    backstory='You are a labor-market analyst who summarizes trends clearly and practically.',
    verbose=True,
    llm='openai/gpt-4o-mini'
)

skills_gap_analyst = Agent(
    role='Skills Gap Analyst',
    goal='Identify strengths and missing capabilities between candidate background and {target_role}.',
    backstory='You are a technical mentor focused on realistic, high-impact upskilling plans.',
    verbose=True,
    llm='openai/gpt-4o-mini'
)

roadmap_writer = Agent(
    role='Career Roadmap Planner',
    goal='Create a concrete 90-day transition plan with weekly milestones and portfolio work.',
    backstory='You build practical plans with measurable milestones and time-boxed learning tasks.',
    verbose=True,
    llm='openai/gpt-4o-mini'
)


## 5. Define Tasks


In [ ]:
research_task = Task(
    description=(
        'Research the demand for {target_role} in {region}. '        'Summarize required skills, common responsibilities, and market outlook.'
    ),
    expected_output='A concise market analysis report with bullet points and practical insights.',
    agent=market_researcher
)

gap_task = Task(
    description=(
        'Using {background} and the research context, identify skill gaps for {target_role}. '        'Prioritize gaps as high/medium/low.'
    ),
    expected_output='A prioritized skill-gap analysis with rationale.',
    context=[research_task],
    agent=skills_gap_analyst
)

plan_task = Task(
    description=(
        'Create a 90-day transition plan for {target_role} with weekly actions, '        'portfolio project ideas, and interview-prep steps.'
    ),
    expected_output='A practical 90-day roadmap with weekly milestones and outcomes.',
    context=[gap_task],
    agent=roadmap_writer
)


## 6. Build and Run the Crew


In [ ]:
career_pivot_crew = Crew(
    agents=[market_researcher, skills_gap_analyst, roadmap_writer],
    tasks=[research_task, gap_task, plan_task],
    process=Process.sequential,
    verbose=True
)

result = career_pivot_crew.kickoff(inputs=inputs)
print(result)


## 7. Optional: Save Result


In [ ]:
from pathlib import Path

out_dir = Path('3_crew/community_contributions/eddy_mmaitsimwale/output')
out_dir.mkdir(parents=True, exist_ok=True)
output_path = out_dir / 'career_pivot_plan.md'
output_path.write_text(str(result), encoding='utf-8')

print(f'Saved result to: {output_path}')
